# Assignment 2: training pipeline

You're using a notebook for this stage. If you'd rather use a script,
write `train.py` instead and delete this file. `main.py` refuses to run if
it finds both `train.py` and `train.ipynb`, so exactly one of them must exist.

Replace this cell and everything below it with your own code. When run
via `uv run python main.py train`, this notebook must, start to finish, with no
manual steps in between:

- Get your Roboflow API key via `from api_key import load_api_key`, so that it
  works whether the key is in your `.env` file or in the environment, and
  download your dataset version from Roboflow Universe.
- Finetune yolo11n/yolo11s (or the -cls variant) on that dataset, from the
  pretrained checkpoint.
- Evaluate on the test split against the two baselines named in the brief: the
  pretrained COCO model zero-shot, and a trivial predict-nothing / majority
  class baseline.
- Sweep the confidence threshold, compute the expected cost at each point from
  the cost matrix you defined in REPORT.md, and save the sweep.
- Save everything `app.py` needs to build its dashboard without retraining:
  the metrics, the per-class numbers, the cost sweep, and the chosen operating
  point.

This is your real training pipeline. It takes as long as it takes, and it runs
wherever training is practical for you: your own machine, a GPU you have access
to, or a hosted notebook. Nothing re-runs it to check your submission.

The weights it produces are what you commit to `weights/` and what your app
serves. Say in REPORT.md where you trained and what settings you used.

Save the trained model into `weights/`. That is the file you commit and the
file your app loads.

How you structure the code beyond that is your call to make and be able to
defend: file layout, function boundaries, naming.


In [ ]:
import json
from collections import Counter
from pathlib import Path

import numpy as np
import yaml
from roboflow import Roboflow

from api_key import load_api_key

WORKSPACE, PROJECT, VERSION = "joseph-nelson", "hard-hat-workers", 10
DATA_DIR = Path("data")
OUT_DIR = Path("outputs")
SPLITS = ["train", "valid", "test"]
IMG_EXT = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}  # ignore .npy cache files


def download_dataset() -> Path:
    ds_dir = DATA_DIR / f"{PROJECT}-v{VERSION}"
    if not (ds_dir / "data.yaml").exists():
        rf = Roboflow(api_key=load_api_key())
        version = rf.workspace(WORKSPACE).project(PROJECT).version(VERSION)
        version.download("yolov11", location=str(ds_dir))
    return ds_dir


def describe_dataset(ds_dir: Path) -> dict:
    names = yaml.safe_load((ds_dir / "data.yaml").read_text())["names"]
    if isinstance(names, dict):
        names = [names[i] for i in sorted(names)]
    stats = {"classes": names, "splits": {}}
    for split in SPLITS:
        images = [p for p in (ds_dir / split / "images").iterdir()
                  if p.suffix.lower() in IMG_EXT]
        instances, images_with = Counter(), Counter()
        areas = {n: [] for n in names}
        for img in images:
            label = ds_dir / split / "labels" / f"{img.stem}.txt"
            seen = set()
            if label.exists():
                for line in label.read_text().splitlines():
                    parts = line.split()
                    if len(parts) < 5:
                        continue
                    name = names[int(parts[0])]
                    instances[name] += 1
                    seen.add(name)
                    areas[name].append(float(parts[3]) * float(parts[4]))
            for name in seen:
                images_with[name] += 1
        stats["splits"][split] = {
            "images": len(images),
            "instances": {n: instances[n] for n in names},
            "images_with_class": {n: images_with[n] for n in names},
            "median_box_area_frac": {
                n: float(np.median(a)) if a else None for n, a in areas.items()
            },
        }
    return stats

In [ ]:
OUT_DIR.mkdir(exist_ok=True)
ds_dir = download_dataset()
stats = describe_dataset(ds_dir)
(OUT_DIR / "dataset_stats.json").write_text(json.dumps(stats, indent=2))
print(json.dumps(stats, indent=2))

{
  "classes": [
    "head",
    "helmet",
    "person"
  ],
  "splits": {
    "train": {
      "images": 4916,
      "instances": {
        "head": 4612,
        "helmet": 13919,
        "person": 450
      },
      "images_with_class": {
        "head": 911,
        "helmet": 4503,
        "person": 154
      },
      "median_box_area_frac": {
        "head": 0.005943270002675943,
        "helmet": 0.00822289156626506,
        "person": 0.08157580138332222
      }
    },
    "valid": {
      "images": 1413,
      "instances": {
        "head": 1339,
        "helmet": 3913,
        "person": 101
      },
      "images_with_class": {
        "head": 256,
        "helmet": 1297,
        "person": 32
      },
      "median_box_area_frac": {
        "head": 0.006174318030605456,
        "helmet": 0.008648648648648649,
        "person": 0.06203703703703704
      }
    },
    "test": {
      "images": 706,
      "instances": {
        "head": 726,
        "helmet": 1915,
        "person": 6

: 

In [ ]:
import os
import shutil
import time

import torch
from ultralytics import YOLO

MODEL = "yolo11s.pt"
IMGSZ = int(os.environ.get("IMGSZ", 640))
BATCH = int(os.environ.get("BATCH", 16))
EPOCHS = int(os.environ.get("EPOCHS", 25))
PATIENCE = int(os.environ.get("PATIENCE", 8))
CACHE = os.environ.get("CACHE", "disk")
FRACTION = float(os.environ.get("FRACTION", 1.0))
SKIP_TRAIN = os.environ.get("SKIP_TRAIN", "0") == "1"  # debug only: reuse weights/best.pt
SEED = 0
RUNS_DIR = Path("runs").resolve()
BEST = Path("weights/best.pt")


def make_data_yaml(ds_dir: Path) -> Path:
    # Roboflow's data.yaml uses relative paths; write absolute ones next to the data.
    names = yaml.safe_load((ds_dir / "data.yaml").read_text())["names"]
    cfg = {
        "path": str(ds_dir.resolve()),
        "train": "train/images",
        "val": "valid/images",
        "test": "test/images",
        "names": names,
    }
    out = ds_dir / "data_abs.yaml"
    out.write_text(yaml.safe_dump(cfg))
    return out


def train_model(data_yaml: Path) -> None:
    t0 = time.time()
    model = YOLO(MODEL)
    model.train(
        data=str(data_yaml), epochs=EPOCHS, imgsz=IMGSZ, batch=BATCH,
        seed=SEED, fraction=FRACTION, patience=PATIENCE, cache=CACHE,
        project=str(RUNS_DIR), name="hardhat", exist_ok=True,
    )
    BEST.parent.mkdir(exist_ok=True)
    shutil.copy(RUNS_DIR / "hardhat" / "weights" / "best.pt", BEST)
    for f in ["results.csv", "results.png"]:
        src = RUNS_DIR / "hardhat" / f
        if src.exists():
            shutil.copy(src, OUT_DIR / f"train_{f}")
    epochs_run = sum(1 for _ in open(RUNS_DIR / "hardhat" / "results.csv")) - 1
    settings = {
        "model": MODEL, "imgsz": IMGSZ, "batch": BATCH, "epochs_max": EPOCHS,
        "epochs_run": epochs_run, "patience": PATIENCE, "cache": CACHE,
        "fraction": FRACTION, "seed": SEED,
        "dataset": f"{WORKSPACE}/{PROJECT}/v{VERSION}",
        "device": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu",
        "torch": torch.__version__,
        "ultralytics": __import__("ultralytics").__version__,
        "train_minutes": round((time.time() - t0) / 60, 1),
    }
    (OUT_DIR / "train_settings.json").write_text(json.dumps(settings, indent=2))

New https://pypi.org/project/ultralytics/8.4.190 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.173 🚀 Python-3.12.3 torch-2.11.0+cpu CPU (Intel Xeon Platinum 8370C CPU @ 2.80GHz)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/workspaces/daiist-assignment-2/outputs/data_abs.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, 

In [ ]:
POSITIVE = "head"
THRESHOLDS = np.round(np.arange(0.01, 1.0, 0.01), 2)


def load_costs() -> dict:
    path = Path("costs.json")
    if not path.exists():
        raise FileNotFoundError("Create costs.json (c_fp, c_fn, c_tp, c_tn, base_rate).")
    return json.loads(path.read_text())


def gt_counts(ds_dir: Path, split: str, names: list) -> dict:
    """Ground-truth instances per class for every image of a split."""
    items = {}
    for img in sorted((ds_dir / split / "images").iterdir()):
        if img.suffix.lower() not in IMG_EXT:
            continue
        counts = np.zeros(len(names), dtype=int)
        label = ds_dir / split / "labels" / f"{img.stem}.txt"
        if label.exists():
            for line in label.read_text().splitlines():
                parts = line.split()
                if len(parts) >= 5:
                    counts[int(parts[0])] += 1
        items[img.stem] = (img, counts)
    return items


def image_scores(weights: str, items: dict, names: list) -> np.ndarray:
    """Per image and class: max confidence of a detection with that class name."""
    model = YOLO(weights)
    wanted = {n.lower(): j for j, n in enumerate(names)}
    stems = list(items)
    index = {s: i for i, s in enumerate(stems)}
    scores = np.zeros((len(stems), len(names)))
    paths = [str(items[s][0]) for s in stems]
    for r in model.predict(paths, stream=True, conf=0.001, imgsz=IMGSZ,
                           batch=BATCH, verbose=False):
        i = index[Path(r.path).stem]
        for cls_id, conf in zip(r.boxes.cls.tolist(), r.boxes.conf.tolist()):
            j = wanted.get(model.names[int(cls_id)].lower())
            if j is not None:
                scores[i, j] = max(scores[i, j], conf)
    return scores


def average_precision(y: np.ndarray, s: np.ndarray) -> float:
    if y.sum() == 0:
        return float("nan")
    if np.all(s == s[0]):  # constant score = chance level = prevalence
        return float(y.mean())
    y = y[np.argsort(-s, kind="stable")]
    tp, fp = np.cumsum(y), np.cumsum(1 - y)
    return float(((tp / (tp + fp)) * y).sum() / y.sum())


def cost_curve(y: np.ndarray, s: np.ndarray, costs: dict) -> list:
    """Expected cost per 1,000 cases at each threshold, reweighted to the assumed base rate."""
    pi = costs["base_rate"]
    P, N = y.sum(), (1 - y).sum()
    rows = []
    for t in THRESHOLDS:
        pred = s >= t
        tpr = float((pred & (y == 1)).sum() / P)
        fpr = float((pred & (y == 0)).sum() / N)
        cost = 1000 * (pi * ((1 - tpr) * costs["c_fn"] + tpr * costs["c_tp"])
                       + (1 - pi) * (fpr * costs["c_fp"] + (1 - fpr) * costs["c_tn"]))
        rows.append({"threshold": float(t), "tpr": tpr, "fpr": fpr, "cost_per_1000": cost})
    return rows


def save_scores(path: Path, items: dict, names: list, scores: np.ndarray) -> None:
    header = ["image"] + [f"gt_{n}" for n in names] + [f"score_{n}" for n in names]
    with open(path, "w") as f:
        f.write(",".join(header) + "\n")
        for i, (stem, (img, counts)) in enumerate(items.items()):
            row = [img.name] + [str(c) for c in counts] + [f"{x:.4f}" for x in scores[i]]
            f.write(",".join(row) + "\n")


def run_evaluation(ds_dir: Path, data_yaml: Path, costs: dict) -> dict:
    names = yaml.safe_load((ds_dir / "data.yaml").read_text())["names"]
    if isinstance(names, dict):
        names = [names[i] for i in sorted(names)]
    pos = names.index(POSITIVE)
    out = {"positive_class": POSITIVE, "costs": costs}
    val_args = dict(data=str(data_yaml), split="test", imgsz=IMGSZ, batch=BATCH,
                    iou=0.7, verbose=False, project=str(OUT_DIR.resolve()), exist_ok=True)

    # 1) Detection metrics on test at conf=0.001 (standard for mAP). P and R are
    #    reported at the confidence that maximises F1.
    m = YOLO(str(BEST)).val(conf=0.001, plots=True, name="test_eval", **val_args)
    for f in (OUT_DIR / "test_eval").glob("val_batch*"):
        f.unlink()
    out["detection_test"] = {
        "map50": float(m.box.map50), "map50_95": float(m.box.map),
        "per_class": {
            names[int(c)]: {"precision": float(m.box.p[k]), "recall": float(m.box.r[k]),
                            "ap50": float(m.box.ap50[k]), "ap50_95": float(m.box.ap[k])}
            for k, c in enumerate(m.box.ap_class_index)},
    }

    # 1b) Confusion matrix at a deployable confidence (0.25).
    #     Rows = predicted, columns = true.
    m25 = YOLO(str(BEST)).val(conf=0.25, plots=True, name="test_eval_conf025", **val_args)
    for f in (OUT_DIR / "test_eval_conf025").iterdir():
        if f.is_file() and not f.name.startswith("confusion_matrix"):
            f.unlink()
    out["confusion_matrix_conf025"] = {
        "matrix": m25.confusion_matrix.matrix.tolist(),
        "labels": names + ["background"], "orientation": "rows=predicted, columns=true"}

    # 2) Image-level scores: finetuned on valid and test, COCO zero-shot on test.
    valid_items, test_items = gt_counts(ds_dir, "valid", names), gt_counts(ds_dir, "test", names)
    s_valid = image_scores(str(BEST), valid_items, names)
    s_test = image_scores(str(BEST), test_items, names)
    s_coco = image_scores(MODEL, test_items, names)  # COCO-pretrained, zero-shot
    save_scores(OUT_DIR / "image_scores_valid.csv", valid_items, names, s_valid)
    save_scores(OUT_DIR / "image_scores_test.csv", test_items, names, s_test)
    gt_valid = np.array([c for _, c in valid_items.values()])
    gt_test = np.array([c for _, c in test_items.values()])
    y_valid = (gt_valid[:, pos] > 0).astype(int)
    y_test = (gt_test[:, pos] > 0).astype(int)
    out["prevalence"] = {"valid": float(y_valid.mean()), "test": float(y_test.mean()),
                         "assumed_production": costs["base_rate"]}

    # 3) Cost sweep: choose the threshold on valid, report the cost on test.
    sweep_valid = cost_curve(y_valid, s_valid[:, pos], costs)
    sweep_test = cost_curve(y_test, s_test[:, pos], costs)
    best_i = int(np.argmin([r["cost_per_1000"] for r in sweep_valid]))
    for name, rows in [("valid", sweep_valid), ("test", sweep_test)]:
        with open(OUT_DIR / f"cost_sweep_{name}.csv", "w") as f:
            f.write("threshold,tpr,fpr,cost_per_1000\n")
            for r in rows:
                f.write(f"{r['threshold']},{r['tpr']:.5f},{r['fpr']:.5f},{r['cost_per_1000']:.4f}\n")
    pi = costs["base_rate"]
    never = 1000 * (pi * costs["c_fn"] + (1 - pi) * costs["c_tn"])
    always = 1000 * (pi * costs["c_tp"] + (1 - pi) * costs["c_fp"])
    out["operating_point"] = {
        "threshold": sweep_valid[best_i]["threshold"], "selected_on": "valid",
        "cost_per_1000_valid": sweep_valid[best_i]["cost_per_1000"],
        "cost_per_1000_test": sweep_test[best_i]["cost_per_1000"],
        "tpr_test": sweep_test[best_i]["tpr"], "fpr_test": sweep_test[best_i]["fpr"],
        "cost_per_1000_never_alert": never, "cost_per_1000_always_alert": always,
    }

    # 4) Baselines at image level (test).
    coco_names = {v.lower() for v in YOLO(MODEL).names.values()}
    ap = lambda sc: {n: average_precision((gt_test[:, j] > 0).astype(int), sc[:, j])
                     for j, n in enumerate(names)}
    out["baselines_test"] = {
        "finetuned_image_ap": ap(s_test),
        "coco_zero_shot_image_ap": ap(s_coco),
        "coco_zero_shot_matching_classes": [n for n in names if n.lower() in coco_names],
        "trivial_predict_nothing_image_ap": ap(np.zeros_like(s_test)),
        "coco_zero_shot_cost_per_1000": cost_curve(y_test, s_coco[:, pos], costs)[best_i]["cost_per_1000"],
    }
    (OUT_DIR / "metrics.json").write_text(json.dumps(out, indent=2))
    return out

In [ ]:
costs = load_costs()
data_yaml = make_data_yaml(ds_dir)
if SKIP_TRAIN:
    assert BEST.exists(), "SKIP_TRAIN=1 needs weights/best.pt"
else:
    train_model(data_yaml)
results = run_evaluation(ds_dir, data_yaml, costs)
print(json.dumps(results["operating_point"], indent=2))
print(json.dumps(results["detection_test"]["per_class"], indent=2))
print(json.dumps(results["baselines_test"], indent=2))